# GeoMapBench — apply the five additional data-quality fixes

This notebook updates only:

1. `metric_distance_computation` — four distance units
2. `population_density_estimation` — nine reference years
3. `topological_directional_reasoning` — polygon-based visual regions
4. `spatial_graph_construction` — visible RGB imagery and graph overlays
5. `shortest_path_optimization` — visible RGB imagery and route overlays

The two SpaceNet tasks are upgraded from their existing Drive folders. **SpaceNet is not downloaded again.** Existing task folders are replaced only after a validated staging copy passes.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/GeoMapBench_Data')
CODE_ZIP = DRIVE_ROOT / 'GeoMapBench-additional-fixes.zip'
FINAL_OUT = DRIVE_ROOT / 'geomapbench_100'
CACHE = DRIVE_ROOT / 'cache'
LOCAL_OUT = Path('/content/geomapbench_build')
REPO = Path('/content/GeoMapBench')
EXTRACT_ROOT = Path('/content/geomapbench_additional_fixes_extract')

for path in [DRIVE_ROOT, FINAL_OUT, CACHE, LOCAL_OUT]:
    path.mkdir(parents=True, exist_ok=True)

if not CODE_ZIP.is_file():
    from google.colab import files
    print(f'Missing {CODE_ZIP}. Select GeoMapBench-additional-fixes.zip.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Upload exactly one code ZIP.')
    uploaded_name = next(iter(uploaded))
    shutil.move(uploaded_name, CODE_ZIP)

shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f'Corrupt ZIP member: {bad_member}')
    archive.extractall(EXTRACT_ROOT)

candidates = [
    path.parent
    for path in EXTRACT_ROOT.rglob('pyproject.toml')
    if (path.parent / 'geomapbench_data').is_dir()
]
if len(candidates) != 1:
    raise RuntimeError(f'Expected one code root, found: {candidates}')

shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(candidates[0], REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print('Loaded code from:', REPO)
print('Final dataset root:', FINAL_OUT)


In [ ]:
import importlib
import json
import shutil
import subprocess

importlib.invalidate_caches()

from geomapbench_data.common import DATA_REVISION
from geomapbench_data.validate import validate_task

AFFECTED = {
    'metric_distance_computation',
    'population_density_estimation',
    'topological_directional_reasoning',
    'spatial_graph_construction',
    'shortest_path_optimization',
}


def publish_staging(leaf: str, staging_task: Path) -> None:
    final_task = FINAL_OUT / leaf
    backup_task = FINAL_OUT / f'.{leaf}_previous'

    errors = validate_task(staging_task)
    if errors:
        raise RuntimeError(f'{leaf} staging validation failed:\n' + '\n'.join(errors))

    shutil.rmtree(backup_task, ignore_errors=True)
    if final_task.exists():
        final_task.rename(backup_task)
    try:
        staging_task.rename(final_task)
    except Exception:
        if backup_task.exists() and not final_task.exists():
            backup_task.rename(final_task)
        raise
    shutil.rmtree(backup_task, ignore_errors=True)
    print('PUBLISHED:', leaf)
    print('Saved to:', final_task)


def run_and_publish(command: str, leaf: str, arguments: list[str]) -> None:
    if leaf not in AFFECTED:
        raise ValueError(f'Refusing to modify an unrelated task: {leaf}')

    local_task = LOCAL_OUT / leaf
    staging_root = FINAL_OUT / '.incoming_additional_fixes'
    staging_task = staging_root / leaf

    shutil.rmtree(local_task, ignore_errors=True)
    shutil.rmtree(staging_task, ignore_errors=True)
    staging_root.mkdir(parents=True, exist_ok=True)

    cmd = ['geomapbench-data', command, *arguments, '--output', str(LOCAL_OUT)]
    print('\nRUN:', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True)

    errors = validate_task(local_task)
    if errors:
        raise RuntimeError(f'{leaf} local validation failed:\n' + '\n'.join(errors))

    manifest = json.loads((local_task / 'manifest.json').read_text(encoding='utf-8'))
    if manifest.get('data_revision') != DATA_REVISION:
        raise RuntimeError(f"{leaf}: wrong data revision {manifest.get('data_revision')!r}")

    shutil.copytree(local_task, staging_task)
    publish_staging(leaf, staging_task)


## Regenerate the three lightweight public-data tasks

These use Natural Earth and cached World Bank responses. They do not require large imagery downloads.


In [ ]:
run_and_publish(
    'metric-distance',
    'metric_distance_computation',
    ['--cache', str(CACHE)],
)

run_and_publish(
    'population-density',
    'population_density_estimation',
    ['--cache', str(CACHE)],
)

run_and_publish(
    'topology-direction',
    'topological_directional_reasoning',
    ['--cache', str(CACHE)],
)


## Upgrade the two existing SpaceNet tasks without redownloading SpaceNet

The migration reuses the high-bit-depth TIFF files already copied into each task folder. It creates visible RGB PNGs and satellite overlays in a validated staging copy.


In [ ]:
from tqdm.auto import tqdm
from geomapbench_data.network_generators import upgrade_existing_spacenet_visuals


def upgrade_existing_spacenet_and_publish(leaf: str) -> None:
    final_task = FINAL_OUT / leaf
    if not final_task.is_dir():
        raise FileNotFoundError(f'Missing existing task folder: {final_task}')

    staging_root = FINAL_OUT / '.incoming_additional_fixes'
    staging_task = staging_root / leaf
    shutil.rmtree(staging_task, ignore_errors=True)
    staging_root.mkdir(parents=True, exist_ok=True)
    shutil.copytree(final_task, staging_task)

    with tqdm(total=100, desc=f'Upgrading {leaf}', unit='record', dynamic_ncols=True) as bar:
        previous = 0

        def update_progress(completed: int, total: int) -> None:
            nonlocal previous
            if completed > previous:
                bar.update(completed - previous)
                previous = completed
            bar.set_postfix({'total': total})

        upgrade_existing_spacenet_visuals(staging_task, progress_callback=update_progress)

    publish_staging(leaf, staging_task)


upgrade_existing_spacenet_and_publish('spatial_graph_construction')
upgrade_existing_spacenet_and_publish('shortest_path_optimization')


## Validate and summarize all five fixes


In [ ]:
from collections import Counter
from PIL import Image
import numpy as np

failures = []
for leaf in sorted(AFFECTED):
    task_dir = FINAL_OUT / leaf
    errors = validate_task(task_dir)
    if errors:
        failures.extend(f'{leaf}: {error}' for error in errors)
        print('FAILED:', leaf)
    else:
        print('PASSED:', leaf)

if failures:
    raise RuntimeError('Validation failed:\n' + '\n'.join(failures))


def records_for(leaf: str):
    return [
        json.loads(line)
        for line in (FINAL_OUT / leaf / 'data.jsonl').read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]

metric_records = records_for('metric_distance_computation')
population_records = records_for('population_density_estimation')
topology_records = records_for('topological_directional_reasoning')

print('\nDistance units:', dict(sorted(Counter(r['target']['unit_id'] for r in metric_records).items())))
print('Population years:', dict(sorted(Counter(r['target']['year'] for r in population_records).items())))
print('Topology visual geometry:', Counter(r['input']['visual_geometry'] for r in topology_records))

for leaf in ['spatial_graph_construction', 'shortest_path_optimization']:
    records = records_for(leaf)
    first_path = FINAL_OUT / leaf / records[0]['input']['images'][0]
    with Image.open(first_path) as image:
        array = np.asarray(image)
        print(
            f'{leaf}: format={image.format}, mode={image.mode}, size={image.size}, '
            f'mean={array.mean():.1f}, contrast={np.percentile(array, 99) - np.percentile(array, 1):.1f}'
        )

print('\nAll five additional fixes passed.')
